In [ ]:
# Configuration
import os
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *

IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

DB_HOST = os.environ.get('POSTGRES_HOST', 'localhost')
DB_PORT = os.environ.get('POSTGRES_PORT', '5432')
DB_NAME = os.environ.get('POSTGRES_DB', 'flex_finops')
DB_USER = os.environ.get('POSTGRES_USER', 'flex_admin')
DB_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'flex_local_dev_2026')

JDBC_URL = f'jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}?stringtype=unspecified'
JDBC_PROPS = {
    'user': DB_USER,
    'password': DB_PASSWORD,
    'driver': 'org.postgresql.Driver',
    'currentSchema': 'flex',
    'stringtype': 'unspecified',
}

# Anomaly detection parameters
ROLLING_WINDOW_DAYS = 14  # Look back 14 days for baseline
Z_SCORE_THRESHOLD = 2.0   # Flag if > 2 standard deviations
MIN_COST_THRESHOLD = 100  # Ignore anomalies below $100 impact

if not IS_DATABRICKS:
    spark = (SparkSession.builder
        .master('local[*]')
        .appName('Flex-Anomaly-Detection')
        .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.1')
        .config('spark.driver.memory', '4g')
        .getOrCreate())
else:
    spark = SparkSession.builder.getOrCreate()

print(f'Anomaly Detection Pipeline')
print(f'  Rolling window: {ROLLING_WINDOW_DAYS} days')
print(f'  Z-score threshold: {Z_SCORE_THRESHOLD}')
print(f'  Min cost threshold: ${MIN_COST_THRESHOLD}')

In [ ]:
# Read cloud_usage from PostgreSQL (or from local transformed data)
try:
    usage_df = spark.read.jdbc(JDBC_URL, 'flex.cloud_usage', properties=JDBC_PROPS)
    print(f'Read {usage_df.count()} rows from flex.cloud_usage')
except Exception as e:
    print(f'DB read failed: {e}')
    print('Falling back to local parquet...')
    import pandas as pd
    data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'cur_raw')
    # Read raw and aggregate daily
    raw = spark.read.parquet(data_dir)
    usage_df = (raw
        .withColumn('time', F.to_date('lineItem/UsageStartDate'))
        .withColumnRenamed('custom/BusinessUnitId', 'business_unit_id')
        .withColumnRenamed('lineItem/ProductCode', 'service')
        .groupBy('business_unit_id', 'time', 'service')
        .agg(F.sum(F.col('lineItem/UnblendedCost').cast('double')).alias('unblended_cost'))
    )
    print(f'Aggregated {usage_df.count()} daily service records from raw data')

usage_df.show(5)

In [ ]:
# ===========================
# Compute Rolling Statistics
# ===========================

# Window: per BU + service, ordered by time, last N days
window_spec = (
    Window
    .partitionBy('business_unit_id', 'service')
    .orderBy(F.col('time').cast('long'))
    .rangeBetween(-ROLLING_WINDOW_DAYS * 86400, -86400)  # Exclude current day
)

# Aggregate daily cost per BU + service (if not already)
daily_cost = (usage_df
    .groupBy('business_unit_id', 'time', 'service')
    .agg(F.sum('unblended_cost').alias('daily_cost'))
)

# Add rolling mean and stddev
stats_df = (daily_cost
    .withColumn('rolling_mean', F.avg('daily_cost').over(window_spec))
    .withColumn('rolling_std', F.stddev('daily_cost').over(window_spec))
    .withColumn('rolling_std', F.coalesce(F.col('rolling_std'), F.lit(0.0)))
)

# Compute Z-score
stats_df = stats_df.withColumn(
    'z_score',
    F.when(F.col('rolling_std') > 0,
           (F.col('daily_cost') - F.col('rolling_mean')) / F.col('rolling_std')
    ).otherwise(0.0)
)

print('Rolling statistics computed')
stats_df.filter(F.col('z_score') > 1.5).show(5, truncate=False)

In [ ]:
# ===========================
# Detect Anomalies (Z-score > threshold)
# ===========================

anomalies_df = (stats_df
    .filter(F.col('z_score') > Z_SCORE_THRESHOLD)
    .filter(F.col('daily_cost') > MIN_COST_THRESHOLD)
    .filter(F.col('rolling_mean').isNotNull())  # Need baseline to compare
)

# Compute severity based on z-score
anomalies_df = anomalies_df.withColumn('severity',
    F.when(F.col('z_score') > 4.0, F.lit('critical'))
    .when(F.col('z_score') > 3.0, F.lit('high'))
    .when(F.col('z_score') > 2.5, F.lit('medium'))
    .otherwise(F.lit('low'))
)

# Compute impact
anomalies_df = anomalies_df.withColumn(
    'impact_amount', F.col('daily_cost') - F.col('rolling_mean')
)

# Compute delta percent
anomalies_df = anomalies_df.withColumn(
    'delta_percent',
    F.round(((F.col('daily_cost') - F.col('rolling_mean')) / F.col('rolling_mean')) * 100, 1)
)

# Build title
anomalies_df = anomalies_df.withColumn('title',
    F.concat(
        F.lit('Cost spike — '), F.col('service'),
        F.lit(' (+'), F.format_number('delta_percent', 1), F.lit('%)')
    )
)

print(f'\n🚨 Anomalies detected: {anomalies_df.count()}')
anomalies_df.select('business_unit_id', 'time', 'service', 'severity', 'daily_cost', 'rolling_mean', 'z_score', 'delta_percent').show(10, truncate=False)

In [ ]:
# ===========================
# Write Anomalies to PostgreSQL
# ===========================

if anomalies_df.count() > 0:
    anomalies_load = (anomalies_df
        .withColumn('detected_at', F.col('time').cast('timestamp'))
        .withColumn('status', F.lit('open'))
        .withColumn('impact', F.concat(
            F.lit('+$'), F.format_number('impact_amount', 0), F.lit(' projected daily')
        ))
        .withColumn('details', F.to_json(F.struct(
            'z_score', 'rolling_mean', 'rolling_std', 'daily_cost'
        )))
        .select(
            'business_unit_id', 'title', 'severity', 'service',
            'detected_at', 'impact', 'status', 'delta_percent', 'details'
        )
    )

    try:
        anomalies_load.write.mode('append').jdbc(JDBC_URL, 'flex.anomalies', properties=JDBC_PROPS)
        print(f'✅ {anomalies_load.count()} anomalies written to flex.anomalies')
    except Exception as e:
        print(f'DB write failed: {e}')
        # Save locally instead
        out_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'anomalies')
        os.makedirs(out_path, exist_ok=True)
        anomalies_load.toPandas().to_parquet(f'{out_path}/anomalies_detected.parquet')
        print(f'Saved locally to {out_path}/anomalies_detected.parquet')
else:
    print('ℹ️ No anomalies detected in this run')

In [ ]:
# ===========================
# Emit Events (for downstream handlers)
# ===========================
import json
from datetime import datetime

if anomalies_df.count() > 0:
    # Create event records for each anomaly
    events = []
    for row in anomalies_df.select('business_unit_id', 'title', 'severity', 'service', 'delta_percent').collect():
        events.append({
            'business_unit_id': row['business_unit_id'],
            'event_type': 'anomaly.detected',
            'payload': json.dumps({
                'title': row['title'],
                'severity': row['severity'],
                'service': row['service'],
                'delta_percent': row['delta_percent'],
                'detected_at': datetime.now().isoformat(),
            }),
            'processed': False,
        })

    events_df = spark.createDataFrame(events)

    try:
        events_df.write.mode('append').jdbc(JDBC_URL, 'flex.events', properties=JDBC_PROPS)
        print(f'✅ {len(events)} anomaly.detected events emitted to flex.events')
    except Exception as e:
        print(f'Event emission failed (DB): {e}')
        print('Events would go to EventBridge in production')

print('\n🎉 Anomaly detection pipeline complete!')
print(f'   Total anomalies: {anomalies_df.count()}')
print(f'   Critical: {anomalies_df.filter(F.col("severity")=="critical").count()}')
print(f'   High: {anomalies_df.filter(F.col("severity")=="high").count()}')
print(f'   Medium: {anomalies_df.filter(F.col("severity")=="medium").count()}')